# GeoAI Urban Planning Framework - Full Pipeline

**Authors:** Desmond Lartey & Kris M.Y. Law  
**Paper:** 

---

## Pipeline Overview

| Step | Cell | Description | Skip if... |
|------|------|-------------|------------|
| 0 | 1–2 | Install packages + mount Google Drive | Drive already mounted |
| 1 | 3 | Discover valid states (NAIP + buildings) | - |
| 2 | 4 | Set up training tile folder structure | Tiles already created |
| 3 | 5 | Create training tiles (512×512, stride 256) | **Skip** if tiles exist in Drive |
| 4 | 6 | Merge per-state tiles into global training set | **Skip** if merged folders exist |
| 5 | 7 | Train U-Net segmentation model (ResNet-34) | **Skip** if best_model.pth exists |
| 6 | 8 | Evaluate model performance (IoU, F1, etc.) | **Skip** if metrics already saved |
| 7 | 9 | Run inference - generate building masks | **Skip** if prediction .tifs exist |
| 8 | 10 | Spatial aggregation - grid, indicators, PQI, risk | **Skip** if _grid.geojson files exist |
| 9 | 11 | Urban diagnostics, exports, and map visualisation | Run for any state |
| 10 | 12 | Internal construct validation (Pearson + Spearman) | After Cell 11 |
| 11 | 13 | Expert validation pack - 8 experts, 20 cells | After Cell 11 |
| 12 | 14 | Download validation outputs as zip | After Cell 13 |

---

## Required Google Drive folder structure
```
MyDrive/
├── NAIP_50_STATES/          ← NAIP_{STATE}.tif files
├── BUILDINGS_50_STATES/     ← buildings_{STATE}.geojson files
└── planner/
    ├── training_tiles/      ← created by Cell 5
    ├── unet_models/         ← created by Cell 7 (best_model.pth saved here)
    ├── predictions/         ← created by Cell 9
    ├── analysis_outputs/    ← created by Cell 10
    ├── analysis_csv/        ← created by Cell 10
    ├── analysis_rasters/    ← created by Cell 11
    └── validation_outputs/  ← created by Cell 13
```

## Step 0 - Environment Setup
Run once per Colab session. Installs required packages and mounts Google Drive.

In [ ]:
# Install required packages - run once per Colab session
%pip install geoai-py osmnx rasterstats shapely scikit-image --quiet

In [ ]:
# Mount Google Drive - safe to run even if already mounted
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted - skipping')

BASE_DIR = '/content/drive/MyDrive'
print(f'Working from: {BASE_DIR}')

## Step 1 - Discover Valid States
Identifies U.S. states that have both NAIP imagery and building footprint files available. Sets `TEST_STATES` used throughout the pipeline.

In [ ]:
import os

NAIP_DIR = os.path.join(BASE_DIR, 'NAIP_50_STATES')
BLD_DIR  = os.path.join(BASE_DIR, 'BUILDINGS_50_STATES')

naip_states = {
    f.replace('NAIP_', '').replace('.tif', '')
    for f in os.listdir(NAIP_DIR)
    if f.startswith('NAIP_') and f.endswith('.tif')
}
bld_states = {
    f.replace('buildings_', '').replace('.geojson', '')
    for f in os.listdir(BLD_DIR)
    if f.startswith('buildings_') and f.endswith('.geojson')
}

TEST_STATES = sorted(list(naip_states.intersection(bld_states)))

print(f'States with both NAIP imagery and building data: {len(TEST_STATES)}')
print(TEST_STATES)

## Step 2 - Training Tile Folder Structure
Creates the output directory for per-state training tiles. Safe to run even if folders already exist.

In [ ]:
PLANNER_DIR      = os.path.join(BASE_DIR, 'planner')
TRAIN_TILES_ROOT = os.path.join(PLANNER_DIR, 'training_tiles')
GLOBAL_IMG_DIR   = os.path.join(TRAIN_TILES_ROOT, 'images')
GLOBAL_LBL_DIR   = os.path.join(TRAIN_TILES_ROOT, 'labels')

os.makedirs(TRAIN_TILES_ROOT, exist_ok=True)
os.makedirs(GLOBAL_IMG_DIR,   exist_ok=True)
os.makedirs(GLOBAL_LBL_DIR,   exist_ok=True)

print(f'Training tiles root: {TRAIN_TILES_ROOT}')

## Step 3 - Create Training Tiles
 **SKIP THIS CELL if tiles already exist in Drive** (`planner/training_tiles/{STATE}/` folders are populated).

Generates 512×512 pixel image–label tile pairs from NAIP imagery and building footprint data for each state. Stride of 256 pixels creates overlapping tiles to increase sample diversity.

In [ ]:
import geoai
import gc

print('Creating training tiles across states...\n')

for state in TEST_STATES:
    try:
        print(f'→ Sampling tiles from {state}')

        raster_path = os.path.join(NAIP_DIR, f'NAIP_{state}.tif')
        vector_path = os.path.join(BLD_DIR,  f'buildings_{state}.geojson')
        out_dir     = os.path.join(TRAIN_TILES_ROOT, state)
        os.makedirs(out_dir, exist_ok=True)

        tiles = geoai.export_geotiff_tiles(
            in_raster    = raster_path,
            out_folder   = out_dir,
            in_class_data= vector_path,
            tile_size    = 512,
            stride       = 256,
            buffer_radius= 0,
        )
        print(f'  ✓ {state}: {len(tiles)} tiles created')
        del tiles
        gc.collect()

    except Exception as e:
        print(f'  Skipping {state}: {e}')

## Step 4 - Merge Per-State Tiles into Global Training Set
 **SKIP THIS CELL if global `images/` and `labels/` folders are already populated.**

Copies all per-state tiles into shared `images/` and `labels/` folders. State name is prepended to each filename to prevent collisions.

In [ ]:
import shutil

print('Merging tiles into global training folders...\n')

for state in TEST_STATES:
    state_dir = os.path.join(TRAIN_TILES_ROOT, state)
    img_dir   = os.path.join(state_dir, 'images')
    lbl_dir   = os.path.join(state_dir, 'labels')

    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f'  Skipping {state}: missing images or labels folder')
        continue

    for f in os.listdir(img_dir):
        shutil.copy(os.path.join(img_dir, f), os.path.join(GLOBAL_IMG_DIR, f'{state}_{f}'))
    for f in os.listdir(lbl_dir):
        shutil.copy(os.path.join(lbl_dir, f), os.path.join(GLOBAL_LBL_DIR, f'{state}_{f}'))

print(f'Global training dataset ready')
print(f'  Images: {len(os.listdir(GLOBAL_IMG_DIR))}')
print(f'  Labels: {len(os.listdir(GLOBAL_LBL_DIR))}')

## Step 5 - Train U-Net Segmentation Model
 **SKIP THIS CELL if `planner/unet_models/best_model.pth` already exists in Drive.**

Trains a U-Net with ResNet-34 backbone (pretrained on ImageNet) for binary building segmentation.
- Architecture: U-Net + ResNet-34 encoder
- Classes: 2 (background=0, building=1)
- Training: 80% tiles | Validation: 20% tiles
- Batch size: 8 | Epochs: 5 | Learning rate: 0.001
- Saved model: `planner/unet_models/best_model.pth`

In [ ]:
import geoai

UNET_DIR = os.path.join(BASE_DIR, 'planner', 'unet_models')
os.makedirs(UNET_DIR, exist_ok=True)

print('Training U-Net model...')

geoai.train_segmentation_model(
    images_dir      = GLOBAL_IMG_DIR,
    labels_dir      = GLOBAL_LBL_DIR,
    output_dir      = UNET_DIR,
    architecture    = 'unet',
    encoder_name    = 'resnet34',
    encoder_weights = 'imagenet',
    num_channels    = 3,
    num_classes     = 2,
    batch_size      = 8,
    num_epochs      = 5,
    learning_rate   = 0.001,
    val_split       = 0.2,
    verbose         = True,
)

print(f'Model saved to: {UNET_DIR}')

## Step 6 - Evaluate Model Performance
 **SKIP THIS CELL if `model_performance_metrics.csv` already exists in `analysis_csv/`.**

Runs the saved model on the held-out validation tiles (20% of total, same random seed as training).
Reports IoU, F1-score, Precision, Recall, and Pixel Accuracy for the building class.

**Note:** This cell may take 5–15 minutes depending on tile count and hardware.

In [ ]:
import geoai
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from tempfile import NamedTemporaryFile

UNET_DIR   = os.path.join(BASE_DIR, 'planner', 'unet_models')
MODEL_PATH = os.path.join(UNET_DIR, 'best_model.pth')
OUT_CSV    = os.path.join(BASE_DIR, 'planner', 'analysis_csv')
os.makedirs(OUT_CSV, exist_ok=True)

# Load validation tiles (reproducible 20% split - same seed as training)
img_files = sorted([f for f in os.listdir(GLOBAL_IMG_DIR) if f.endswith('.tif')])
lbl_files = sorted([f for f in os.listdir(GLOBAL_LBL_DIR) if f.endswith('.tif')])
np.random.seed(42)
n_total = len(img_files)
n_val   = max(1, int(n_total * 0.2))
val_idx = np.random.choice(n_total, size=n_val, replace=False)
val_imgs = [os.path.join(GLOBAL_IMG_DIR, img_files[i]) for i in val_idx]
val_lbls = [os.path.join(GLOBAL_LBL_DIR, lbl_files[i]) for i in val_idx]
print(f'Validation tiles: {n_val} of {n_total} ({n_val/n_total*100:.1f}% hold-out)')

def compute_metrics(model_path, img_paths, lbl_paths):
    TP = FP = FN = TN = 0
    for i, (img_path, lbl_path) in enumerate(zip(img_paths, lbl_paths)):
        try:
            with NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
                pred_path = tmp.name
            geoai.semantic_segmentation(
                input_path=img_path, output_path=pred_path,
                model_path=model_path, architecture='unet',
                encoder_name='resnet34', num_channels=3, num_classes=2,
                window_size=512, overlap=256, batch_size=1,
            )
            with rasterio.open(pred_path) as s: pred = s.read(1).astype(np.int32)
            with rasterio.open(lbl_path)  as s: lbl  = s.read(1).astype(np.int32)
            if pred.shape != lbl.shape:
                from skimage.transform import resize
                pred = (resize(pred, lbl.shape, order=0, preserve_range=True) > 0.5).astype(np.int32)
            pred_b = (pred == 1).astype(np.int32)
            lbl_b  = (lbl  == 1).astype(np.int32)
            TP += int(np.sum((pred_b==1)&(lbl_b==1)))
            FP += int(np.sum((pred_b==1)&(lbl_b==0)))
            FN += int(np.sum((pred_b==0)&(lbl_b==1)))
            TN += int(np.sum((pred_b==0)&(lbl_b==0)))
            os.unlink(pred_path)
            if (i+1) % 10 == 0: print(f'  Processed {i+1}/{len(img_paths)} tiles...')
        except Exception as e:
            print(f'  Skipping tile {i}: {e}')
    precision = TP/(TP+FP+1e-9); recall = TP/(TP+FN+1e-9)
    iou = TP/(TP+FP+FN+1e-9); f1 = 2*precision*recall/(precision+recall+1e-9)
    accuracy = (TP+TN)/(TP+FP+FN+TN+1e-9)
    return {'IoU': round(iou,4), 'F1': round(f1,4), 'Precision': round(precision,4),
            'Recall': round(recall,4), 'Accuracy': round(accuracy,4)}

print('\nRunning evaluation (this may take several minutes)...')
metrics = compute_metrics(MODEL_PATH, val_imgs, val_lbls)

print('\n── Model Performance - Building Class (Validation Set) ──')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

# Save metrics CSV
pd.DataFrame([{'Metric': k, 'Value': v} for k, v in metrics.items()]).to_csv(
    os.path.join(OUT_CSV, 'model_performance_metrics.csv'), index=False)

# Save lollipop chart
metric_names = list(metrics.keys())
metric_vals  = list(metrics.values())
colors_lp    = ['#0A6FA8','#B45C0A','#6B2AA8','#2E8B57','#607D8B']
fig, ax = plt.subplots(figsize=(8, 5), facecolor='white')
for i, (label, val, col) in enumerate(zip(metric_names, metric_vals, colors_lp)):
    ax.plot([0, val], [i, i], color=col, linewidth=2.5, solid_capstyle='round')
    ax.scatter(val, i, color=col, s=160, zorder=3, edgecolors='white', linewidths=1.5)
    ax.text(val+0.015, i, f'{val:.3f}', va='center', fontsize=11, fontweight='bold', color=col)
for x in [0.25, 0.5, 0.75, 1.0]:
    ax.axvline(x, color='#E2E8F0', linewidth=0.8)
ax.set_yticks(range(len(metric_names)))
ax.set_yticklabels(metric_names, fontsize=12)
ax.set_xlim(0, 1.15)
ax.set_xlabel('Score', fontsize=12)
ax.set_title('U-Net Segmentation Performance - Building Class\n(Validation Set, 20% hold-out)', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_CSV, 'model_performance_chart.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'\nSaved metrics and chart to: {OUT_CSV}')

## Step 7 - Run Inference (Building Mask Prediction)
 **SKIP THIS CELL if `{STATE}_buildings_pred.tif` files already exist in `planner/predictions/`.**

Applies the trained U-Net model to each state's NAIP imagery using a sliding window approach.
- Window: 512×512 pixels | Overlap: 256 pixels
- Output: binary building mask (.tif) per state → `planner/predictions/`
- Note: roads are NOT produced here - road networks are obtained from OpenStreetMap in Step 8.

In [ ]:
import geoai, gc

MODEL_PATH = os.path.join(BASE_DIR, 'planner', 'unet_models', 'best_model.pth')
PRED_DIR   = os.path.join(BASE_DIR, 'planner', 'predictions')
os.makedirs(PRED_DIR, exist_ok=True)

print('Running building mask inference...')

for state in TEST_STATES:
    try:
        output_raster = os.path.join(PRED_DIR, f'{state}_buildings_pred.tif')
        if os.path.exists(output_raster):
            print(f'  {state}: prediction already exists - skipping')
            continue

        print(f'  → Running inference for {state}')
        geoai.semantic_segmentation(
            input_path   = os.path.join(NAIP_DIR, f'NAIP_{state}.tif'),
            output_path  = output_raster,
            model_path   = MODEL_PATH,
            architecture = 'unet',
            encoder_name = 'resnet34',
            num_channels = 3,
            num_classes  = 2,
            window_size  = 512,
            overlap      = 256,
            batch_size   = 2,
        )
        print(f'  ✓ Saved: {output_raster}')
        gc.collect()

    except Exception as e:
        print(f'  Skipping {state}: {e}')

## Step 8 - Spatial Aggregation: Grid, Indicators, and Building Coverage
 **SKIP THIS CELL if `{STATE}_grid.geojson` files already exist in `planner/analysis_outputs/`.**

For each state:
1. Creates a 100m×100m grid over the study extent (PIX_RES_FACTOR=100 × raster resolution)
2. Extracts building coverage from the segmentation mask
3. Extracts vegetation coverage from NAIP (NDVI if 4-band, ExG proxy if RGB-only)
4. Retrieves road network from OpenStreetMap via OSMnx
5. Saves per-state grid GeoJSON to `planner/analysis_outputs/`

In [ ]:
import rasterio, rasterio.mask
import geopandas as gpd
import numpy as np
import pandas as pd
import os, gc
from shapely.geometry import box, Polygon
import osmnx as ox

PRED_DIR     = os.path.join(BASE_DIR, 'planner', 'predictions')
OUT_ANALYSIS = os.path.join(BASE_DIR, 'planner', 'analysis_outputs')
os.makedirs(OUT_ANALYSIS, exist_ok=True)

PIX_RES_FACTOR = 100  # grid cell = 100 × raster pixel resolution (~100m)

def compute_ndvi(r, n): return (n - r) / (n + r + 1e-6)
def compute_exg(r, g, b): return 2*g - r - b

for state in TEST_STATES:
    out_file = os.path.join(OUT_ANALYSIS, f'{state}_grid.geojson')
    if os.path.exists(out_file):
        print(f'{state}: grid already exists - skipping')
        continue

    print(f'\nProcessing {state}...')
    try:
        test_raster = os.path.join(NAIP_DIR, f'NAIP_{state}.tif')
        seg_raster  = os.path.join(PRED_DIR, f'{state}_buildings_pred.tif')

        # Define study extent and OSM road network
        with rasterio.open(test_raster) as src:
            bounds = src.bounds; crs = src.crs
        bbox_gdf = gpd.GeoDataFrame(geometry=[box(*bounds)], crs=crs).to_crs(epsg=4326)
        lon_min, lat_min, lon_max, lat_max = bbox_gdf.total_bounds
        poly = Polygon([(lon_min,lat_min),(lon_min,lat_max),(lon_max,lat_max),(lon_max,lat_min)])
        G = ox.graph_from_polygon(poly, network_type='drive')
        _, edges = ox.graph_to_gdfs(G)

        # Build 100m grid from segmentation raster bounds
        with rasterio.open(seg_raster) as src:
            seg_bounds = src.bounds; seg_res = src.res[0]; seg_crs = src.crs
        step = seg_res * PIX_RES_FACTOR
        xmin, ymin, xmax, ymax = seg_bounds
        grid_cells = [box(x, y, x+step, y+step)
                      for x in np.arange(xmin, xmax, step)
                      for y in np.arange(ymin, ymax, step)]
        grid = gpd.GeoDataFrame(geometry=grid_cells, crs=seg_crs)

        # Extract building and vegetation coverage per cell
        stats = []
        with rasterio.open(seg_raster) as src_seg, rasterio.open(test_raster) as src_full:
            has_nir = src_full.count >= 4
            for geom in grid.geometry:
                try:
                    seg_arr, _ = rasterio.mask.mask(src_seg, [geom], crop=True)
                    bld_cov = np.count_nonzero(seg_arr[0]==1) / seg_arr[0].size
                    full_arr, _ = rasterio.mask.mask(src_full, [geom], crop=True)
                    r, g, b = [full_arr[i].astype('float32') for i in range(3)]
                    if has_nir:
                        nir = full_arr[3].astype('float32')
                        veg_cov = float(np.mean(compute_ndvi(r, nir) > 0.2))
                    else:
                        exg = compute_exg(r, g, b)
                        exg_n = (exg-exg.min())/(exg.max()-exg.min()+1e-6)
                        veg_cov = float(np.mean(exg_n > 0.3))
                    stats.append({'bld_coverage': bld_cov, 'veg_coverage': veg_cov})
                except:
                    stats.append({'bld_coverage': 0.0, 'veg_coverage': 0.0})

        grid = pd.concat([grid, pd.DataFrame(stats)], axis=1)
        grid.to_crs(epsg=4326).to_file(out_file, driver='GeoJSON')
        print(f'  ✓ Saved: {out_file} ({len(grid)} cells)')
        del grid, stats, edges; gc.collect()

    except Exception as e:
        print(f'  Skipping {state}: {e}')

## Step 9 - Urban Diagnostics, Indicator Computation, and Map Visualisation
Run this cell for any state that has a grid GeoJSON from Step 8.

**Change `VIS_STATE`** to any state code (e.g. `'AL'`, `'CO'`, `'CA'`) to run diagnostics for that state.

This cell computes:
- **Perceptual indicators:** greenness, openness, enclosure, walkability, imageability
- **Perceptual Quality Index (PQI):** weighted composite (weights: 0.25/0.20/0.20/0.20/0.15)
- **Risk indicators:** sprawl score, environmental degradation score, infrastructure deficiency
- **Combined Risk Index:** weighted composite (weights: 0.4/0.3/0.3)
- **Planning recommendations:** rule-based logic from PQI and risk thresholds
- **Outputs:** GeoJSON, CSV, PQI raster, risk raster, interactive map

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import osmnx as ox
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import leafmap

# ── USER SETTING: change state here ──────────────────────────
VIS_STATE = 'AL'   # options: 'AL', 'CO', 'CA', 'AZ', 'AR', etc.
# ─────────────────────────────────────────────────────────────

OUT_ANALYSIS = os.path.join(BASE_DIR, 'planner', 'analysis_outputs')
OUT_RASTERS  = os.path.join(BASE_DIR, 'planner', 'analysis_rasters')
OUT_CSV      = os.path.join(BASE_DIR, 'planner', 'analysis_csv')
for d in [OUT_ANALYSIS, OUT_RASTERS, OUT_CSV]: os.makedirs(d, exist_ok=True)

def minmax(x):
    x = pd.Series(x).astype(float)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

def utm_epsg_from_lonlat(lon, lat):
    return (32600 if lat >= 0 else 32700) + int((lon + 180) / 6) + 1

def rasterize_grid(gdf, col, out_path, res=0.0001):
    bounds = gdf.total_bounds
    w = int((bounds[2]-bounds[0])/res); h = int((bounds[3]-bounds[1])/res)
    transform = from_bounds(*bounds, w, h)
    arr = rasterize([(g, float(v)) for g,v in zip(gdf.geometry, gdf[col])],
                    out_shape=(h,w), transform=transform, fill=np.nan, dtype='float32')
    with rasterio.open(out_path,'w',driver='GTiff',height=h,width=w,
                       count=1,dtype='float32',crs='EPSG:4326',transform=transform) as dst:
        dst.write(arr, 1)
    return bounds

def recommend(row):
    if row['combined_risk'] >= 0.6 and row['PQI'] < 0.4:
        return 'Green corridors; Walkability retrofit; Mixed-use densification'
    if row['infra_deficiency'] > 0.5:
        return 'Improve connectivity; Pedestrian & cycling infrastructure'
    if row['envdeg_score'] > 0.6:
        return 'Urban greening; Blue\u2013green infrastructure'
    return 'Maintain & monitor'

state = VIS_STATE
grid_file   = os.path.join(OUT_ANALYSIS, f'{state}_grid.geojson')
test_raster = os.path.join(BASE_DIR, 'NAIP_50_STATES', f'NAIP_{state}.tif')

if not os.path.exists(grid_file): raise RuntimeError(f'Missing grid: {grid_file}. Run Step 8 first.')
if not os.path.exists(test_raster): raise RuntimeError(f'Missing NAIP: {test_raster}')

grid = gpd.read_file(grid_file).reset_index(drop=True)
grid['cell_id'] = grid.index.astype(int)
print(f'Grid loaded for {state}: {len(grid)} cells')

# Rebuild OSM roads from grid bounding box
grid_wgs84 = grid.to_crs(epsg=4326)
minx, miny, maxx, maxy = grid_wgs84.total_bounds
poly = Polygon([(minx,miny),(minx,maxy),(maxx,maxy),(maxx,miny)])
G = ox.graph_from_polygon(poly, network_type='drive')
_, edges = ox.graph_to_gdfs(G)

# Road density (km/km²) in metric CRS
centroid = grid_wgs84.unary_union.centroid
epsg_utm = utm_epsg_from_lonlat(centroid.x, centroid.y)
grid_m  = grid.to_crs(epsg=epsg_utm)
edges_m = edges.to_crs(epsg=epsg_utm)
grid_m['road_density_km_per_km2'] = grid_m.geometry.apply(
    lambda g: edges_m.clip(g).length.sum()/1000 / (g.area/1e6))
grid['road_density_km_per_km2'] = grid_m['road_density_km_per_km2'].values

# Perceptual indicators
grid['greenness']    = minmax(grid['veg_coverage'])
grid['openness']     = (1 - grid['bld_coverage']).clip(0, 1)
grid['enclosure']    = grid['bld_coverage'].clip(0, 1)
grid['walkability']  = (minmax(grid['road_density_km_per_km2']) * (1 - minmax(grid['enclosure']))).clip(0,1)
grid['imageability'] = minmax(np.sqrt(grid['bld_coverage'].clip(0,1)))

# PQI (weights: greenness 0.25, openness 0.20, enclosure inverse 0.20, walkability 0.20, imageability 0.15)
grid['PQI'] = (0.25*grid['greenness'] + 0.20*grid['openness'] +
               0.20*(1-grid['enclosure']) + 0.20*grid['walkability'] +
               0.15*grid['imageability']).clip(0,1)

# Risk indicators
grid['sprawl_score']     = (1 - grid['PQI']).clip(0, 1)
grid['envdeg_score']     = (1 - grid['greenness']).clip(0, 1)
grid['infra_deficiency'] = ((1-minmax(grid['road_density_km_per_km2']))*(1-grid['walkability'])).clip(0,1)
grid['combined_risk']    = (0.4*grid['sprawl_score'] + 0.3*grid['envdeg_score'] +
                            0.3*grid['infra_deficiency']).clip(0,1)

# Planning recommendations (rule-based logic)
grid['recommendations'] = grid.apply(recommend, axis=1)

# Export vector outputs
grid_wgs84 = grid.to_crs(epsg=4326)
geojson_out = os.path.join(OUT_ANALYSIS, f'{state}_urban_diagnostics.geojson')
csv_out     = os.path.join(OUT_CSV,      f'{state}_urban_grid_results.csv')
grid_wgs84.to_file(geojson_out, driver='GeoJSON')
df_csv = grid_wgs84.copy(); df_csv['geometry'] = df_csv.geometry.astype(str)
df_csv.to_csv(csv_out, index=False)
print(f'Exported: {geojson_out}')
print(f'Exported: {csv_out}')

# Rasterize PQI and risk for map visualisation
PQI_TIF  = os.path.join(OUT_RASTERS, f'{state}_PQI.tif')
RISK_TIF = os.path.join(OUT_RASTERS, f'{state}_combined_risk.tif')
bounds_w = rasterize_grid(grid_wgs84, 'PQI',           PQI_TIF)
_        = rasterize_grid(grid_wgs84, 'combined_risk', RISK_TIF)

# Summary statistics
print(f'\n── Diagnostic Summary for {state} ──')
print(f'  Cells:            {len(grid)}')
print(f'  Mean PQI:         {grid["PQI"].mean():.3f}')
print(f'  Mean Risk:        {grid["combined_risk"].mean():.3f}')
print(f'  High-risk cells:  {(grid["combined_risk"]>0.6).sum()} ({(grid["combined_risk"]>0.6).mean()*100:.1f}%)')
print(f'\nRecommendation distribution:')
print(grid['recommendations'].value_counts().to_string())

del grid_m, edges_m; gc.collect()

# Interactive map
m = leafmap.Map()
m.add_raster(test_raster,  layer_name='NAIP Imagery')
m.add_raster(PQI_TIF,      layer_name='PQI',           colormap='YlGn', opacity=0.8)
m.add_raster(RISK_TIF,     layer_name='Combined Risk',  colormap='OrRd', opacity=0.8)
m.add_geojson(geojson_out, layer_name='Diagnostics (grid)')
m.zoom_to_bounds(list(bounds_w))
m

## Step 10 - Internal Construct Validation
Computes Pearson and Spearman correlation matrices among all framework indicators and composite indices.
Produces heatmap figures and an auto-interpretation text file.

**Requires:** Step 9 output for the selected state (`{STATE}_urban_grid_results.csv`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

STATE        = 'AL'   # match state used in Step 9
OUT_CSV      = os.path.join(BASE_DIR, 'planner', 'analysis_csv')
VALID_DIR    = os.path.join(BASE_DIR, 'planner', 'validation_outputs')
os.makedirs(VALID_DIR, exist_ok=True)

csv_path = os.path.join(OUT_CSV, f'{STATE}_urban_grid_results.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Missing: {csv_path}. Run Step 9 first.')

df = pd.read_csv(csv_path)

vars_to_use = ['bld_coverage','veg_coverage','greenness','enclosure',
               'road_density_km_per_km2','walkability','envdeg_score','PQI','combined_risk']
short_names = {'bld_coverage':'bld_cov','veg_coverage':'veg_cov','greenness':'greenness',
               'enclosure':'enclosure','road_density_km_per_km2':'road_den',
               'walkability':'walkab.','envdeg_score':'envdeg','PQI':'PQI','combined_risk':'comb_risk'}

vars_present = [v for v in vars_to_use if v in df.columns]
df_num = df[vars_present].dropna().rename(columns=short_names)

corr_p = df_num.corr(method='pearson')
corr_s = df_num.corr(method='spearman')

corr_p.to_csv(os.path.join(VALID_DIR, 'construct_validity_corr_pearson.csv'))
corr_s.to_csv(os.path.join(VALID_DIR, 'construct_validity_corr_spearman.csv'))

fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor='white')
shared = dict(annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1,
              linewidths=0.5, linecolor='#E2E8F0', annot_kws={'size':9}, square=True)
sns.heatmap(corr_s, ax=axes[0], **shared, cbar=False)
axes[0].set_title('Spearman Correlation Matrix', fontsize=12, fontweight='bold')
im = sns.heatmap(corr_p, ax=axes[1], **shared)
axes[1].set_title('Pearson Correlation Matrix', fontsize=12, fontweight='bold')
plt.suptitle('Internal Construct Validation - Correlation Matrices', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(VALID_DIR, 'Figure_ConstructValidity_Heatmaps.png'), dpi=300, bbox_inches='tight')
plt.show()

# Key hypothesis checks
pairs = [('bld_cov','enclosure','Strong positive expected'),
         ('veg_cov','greenness','Strong positive expected'),
         ('road_den','walkab.','Positive expected'),
         ('greenness','envdeg','Strong negative expected'),
         ('PQI','comb_risk','Strong negative expected')]
print('── Construct Validity - Key Indicator Relationships ──')
for a, b, note in pairs:
    if a in corr_p.columns and b in corr_p.columns:
        print(f'  {a} vs {b}: Pearson r={corr_p.loc[a,b]:.3f}, Spearman ρ={corr_s.loc[a,b]:.3f} | {note}')
print(f'\nSaved to: {VALID_DIR}')

## Step 11 - Expert Validation Pack (8 Experts)
Generates the expert face validity assessment materials.

**What this cell produces** (saved to `planner/validation_outputs/`):
- `expert_review_pack_20cells.csv` - master pack of 20 randomly sampled grid cells
- `Expert_1/` through `Expert_8/` - one folder per expert, each containing:
  - `ratings_{Expert_X}.csv` - rating sheet with indicator profiles and empty Likert column
  - `INSTRUCTIONS.txt` - rating scale definition, category-level guidance, neutral response protocol
  - `qualifications_{Expert_X}.csv` - professional background form

**Sampling:** 20 cells randomly selected with seed=42 - identical across all experts for inter-rater reliability.

**After experts return files:** fill the Likert scores in the rating CSVs, then run Step 12 to compute agreement statistics.

**Requires:** Step 9 output (`{STATE}_urban_grid_results.csv` or `{STATE}_urban_diagnostics.geojson`).

In [ ]:
import os, re
import numpy as np
import pandas as pd
import geopandas as gpd

STATE         = 'AL'   # must match state used in Step 9
RANDOM_SEED   = 42
N_SAMPLE      = 20
PLANNER_DIR   = os.path.join(BASE_DIR, 'planner')
OUT_ANALYSIS  = os.path.join(PLANNER_DIR, 'analysis_outputs')
OUT_CSV       = os.path.join(PLANNER_DIR, 'analysis_csv')
VALIDATION_DIR = os.path.join(PLANNER_DIR, 'validation_outputs')
os.makedirs(VALIDATION_DIR, exist_ok=True)

EXPERTS = ['Expert_1','Expert_2','Expert_3','Expert_4','Expert_5','Expert_6','Expert_7','Expert_8']

QUALIFICATION_TEMPLATE = pd.DataFrame([{'Field': f, 'Response': ''} for f in [
    'Full name',
    'Professional title / role',
    'Years of experience in urban planning or related field',
    'Primary area of expertise (e.g. land use, transport, environmental, GIS)',
    'Familiarity with AI-based planning tools (None / Basic / Intermediate / Advanced)',
    'Country / region of primary practice',
]])

INSTRUCTIONS = '''RATING INSTRUCTIONS\n====================\nYou are reviewing AI-generated planning recommendations for 20 urban grid cells\nderived from a GeoAI spatial diagnostic framework. Each cell is 100m x 100m.\n\nFor each cell you will see:\n  - Key spatial indicators: building coverage, vegetation coverage, greenness,\n    walkability, road density, PQI (Perceptual Quality Index), combined risk score\n  - A model-generated planning recommendation\n\nYOUR TASK:\nRate each recommendation on a 5-point Likert scale:\n  1 = Strongly disagree - recommendation is inappropriate or misleading for this cell\n  2 = Disagree - recommendation is unlikely to be useful in practice\n  3 = Neutral - recommendation is plausible but you have significant reservations\n  4 = Agree - recommendation is appropriate given the diagnostic evidence\n  5 = Strongly agree - recommendation is clearly appropriate and governance-relevant\n\nTREATMENT OF NEUTRAL RESPONSES (score = 3):\nIf you select 3, please use the notes_optional column to briefly explain\nwhat additional information or context would change your assessment.\n\nCATEGORY-LEVEL GUIDANCE:\n  Connectivity + Active travel    - appropriate when road density is low and walkability is low\n  Maintain / Monitor              - appropriate when PQI is moderate-high and risk is low\n  Green + Walkability + Mixed-use - appropriate when risk is high and PQI is low\n  Greening + Blue-green           - appropriate when greenness is low and envdeg_score is high\n\nPlease complete all 20 rows and return the filled file.'''

def truncate_reco(text):
    if pd.isna(text): return 'None'
    t = str(text).strip()
    mapping = {
        'Improve connectivity; Pedestrian & cycling infrastructure': 'Connectivity + Active travel',
        'Maintain & monitor': 'Maintain / Monitor',
        'Green corridors; Walkability retrofit; Mixed-use densification': 'Green + Walkability + Mixed-use',
        'Urban greening; Blue\u2013green infrastructure': 'Greening + Blue-green',
    }
    return mapping.get(t, t[:55]+'...' if len(t)>58 else t)

# Load diagnostic data (GeoJSON preferred, CSV fallback)
geojson_path = os.path.join(OUT_ANALYSIS, f'{STATE}_urban_diagnostics.geojson')
csv_path     = os.path.join(OUT_CSV,      f'{STATE}_urban_grid_results.csv')

if os.path.exists(geojson_path):
    df = pd.DataFrame(gpd.read_file(geojson_path).drop(columns='geometry'))
elif os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(f'No diagnostic data found for {STATE}. Run Step 9 first.')

reco_col = next((c for c in df.columns if 'recommend' in c.lower()), None)
df['recommendation_short'] = df[reco_col].apply(truncate_reco) if reco_col else 'None'

# Sample 20 cells reproducibly
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.choice(df.index.to_numpy(), size=min(N_SAMPLE, len(df)), replace=False)
review = df.loc[idx].copy().reset_index(drop=False).rename(columns={'index': 'original_index'})
review['cell_id'] = np.arange(1, len(review)+1)

keep_cols = ['cell_id','recommendation_short']
for c in ['bld_coverage','veg_coverage','greenness','road_density_km_per_km2',
           'walkability','PQI','combined_risk','sprawl_score','envdeg_score','infra_deficiency']:
    if c in review.columns: keep_cols.append(c)

expert_pack = review[keep_cols].copy()
num_cols = [c for c in expert_pack.columns if c not in ['cell_id','recommendation_short']]
expert_pack[num_cols] = expert_pack[num_cols].round(3)

# Save master pack
master_path = os.path.join(VALIDATION_DIR, 'expert_review_pack_20cells.csv')
expert_pack.to_csv(master_path, index=False)
print(f'Master pack saved: {master_path}')

# Create one folder per expert
for ex in EXPERTS:
    ex_dir = os.path.join(VALIDATION_DIR, ex)
    os.makedirs(ex_dir, exist_ok=True)

    rating_df = expert_pack.copy()
    rating_df['likert_1to5']   = ''   # expert fills: 1 (Strongly disagree) to 5 (Strongly agree)
    rating_df['notes_optional'] = ''  # required when likert = 3 (Neutral)
    rating_df.to_csv(os.path.join(ex_dir, f'ratings_{ex}.csv'), index=False)

    with open(os.path.join(ex_dir, 'INSTRUCTIONS.txt'), 'w', encoding='utf-8') as f:
        f.write(INSTRUCTIONS)

    QUALIFICATION_TEMPLATE.to_csv(os.path.join(ex_dir, f'qualifications_{ex}.csv'), index=False)

    print(f'  {ex}: folder created')

print(f'\nDone - {len(EXPERTS)} expert folders in: {VALIDATION_DIR}')
print(f'Cells per expert: {N_SAMPLE} | Random seed: {RANDOM_SEED}')
print('\nNext: send each expert their folder. After they return filled ratings, run Step 12.')

## Step 12 - Download Validation Outputs
Zips the entire `validation_outputs` folder and downloads it to your local computer.

Run this after Step 11 (or after receiving completed expert rating files).
The zip will contain all expert folders with rating sheets, instructions, and qualification forms.

In [ ]:
import shutil
from google.colab import files

VALIDATION_DIR = os.path.join(BASE_DIR, 'planner', 'validation_outputs')

shutil.make_archive(
    '/content/validation_outputs',
    'zip',
    os.path.dirname(VALIDATION_DIR),
    os.path.basename(VALIDATION_DIR)
)

files.download('/content/validation_outputs.zip')
print('Download started - check your browser downloads folder.')
print('The zip contains all expert folders with rating sheets, instructions, and qualification forms.')